# Co-registration + signal maps — full record

Generates the three-panel **before/after co-registration + reference→day signal**
figure for every processed date, via
`cntp.postprocessing.coreg_and_signal_figure`. That helper loads each date's
already-written products (coreg M3C2 distances `.npz`, the cached stable
reference, and the `<date>_M3C2_raster.tif` signal) and draws
`cntp.plot.plot_m3c2_coreg_and_signal`.

No SfM / Metashape needed — this only **reads pipeline outputs**, so it's fast
and safe to re-run. Dates missing any required output are skipped. All panels use
a fixed ±4 m scale so the figures are comparable across the whole record.

In [ ]:
%load_ext autoreload
%autoreload 2

import re
import traceback
from pathlib import Path

import site_config as site
from cntp.postprocessing import coreg_and_signal_figure

## Configuration

Edit only this section. The cached stable reference (the corepoint coords) is
auto-discovered from `output/_ref_cache/*_stable.las`, so no downsample
parameter is needed.

In [ ]:
# ── Knobs ────────────────────────────────────────────────────────────
vmax     = 4.0     # shared colour scale ±vmax (m); None = per-figure auto
res      = 1.0     # ground bin size for the coreg panels (m)
save_pdf = False   # PNG only for a big batch; True also writes vector PDFs

# Collect every figure in one folder for easy browsing.
plot_out = site.output_dir / "output" / "coreg_signal_plots"
plot_out.mkdir(parents=True, exist_ok=True)

# Optional inclusive date bounds ("YYYY-MM-DD"); None = no bound (full record).
date_from = None
date_to   = None

## Discover dates that are ready to plot

Scans `output/` for date folders that already have the two inputs this figure
needs — the M3C2 signal raster and the coreg distances `.npz`. Anything still
missing them (unprocessed, or cloud-gated) is skipped.

In [ ]:
out_root = site.output_dir / "output"
_date_re = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def _ready(d: str) -> bool:
    dd = out_root / d
    return ((dd / "single_day" / f"{d}_M3C2_raster.tif").exists()
            and (dd / "coreg" / f"{d}_m3c2_distances.npz").exists())

dates = sorted(
    p.name for p in out_root.iterdir()
    if p.is_dir() and _date_re.match(p.name) and _ready(p.name)
    and (date_from is None or p.name >= date_from)
    and (date_to   is None or p.name <= date_to)
)
print(f"{len(dates)} date(s) ready to plot"
      + (f": {dates[0]} … {dates[-1]}" if dates else "."))
dates

## Generate

One three-panel figure per date → `output/coreg_signal_plots/<date>_coreg_and_signal.png`.
Wrapped in `try/except` so one bad date doesn't stop the batch.

In [ ]:
ok, failed = [], []
for d in dates:
    try:
        coreg_and_signal_figure(
            d, site.output_dir,
            res      = res,
            vmax     = vmax,
            plot_dir = plot_out,
            show     = False,
            save_pdf = save_pdf,
        )
        ok.append(d)
    except Exception as e:
        print(f"  {d} failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        failed.append(d)

print(f"\nDone: {len(ok)} figure(s) → {plot_out}")
if failed:
    print(f"Failed ({len(failed)}): {failed}")